<a href="https://colab.research.google.com/github/jlhelling/s2-dw-wof/blob/main/S2-DW-WOF_workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Global water occurrence frequency (WOF) mapping from Sentinel-2**

This workflow enables you to create *monthly and annual water occurrence frequency (WOF) maps* from full Sentinel-2 time series for any region of interest and export them to Google Drive.

Developed by **Leo Helling**

*UMR 5600 – Environment, Ville, Société (EVS) lab*,
*Centre National de Recherche Scientifique (CNRS)*,
*ENS de Lyon*,
*France*

Please refer to the following publication for any data created from this workflow:

Helling, L., Singh, S., Rey, L., Parmentier, H., Messager, M., Piégay, H., Belletti, B. (2026). *Mapping water in a Dynamic World: Annual Water Occurrence Frequencies from full Sentinel-2 time series* (in preparation)

Contact: leo.helling@ens-lyon.fr

## Step 1: Install required libraries and load functions

In [ ]:
%%capture
!pip install --upgrade ee geemap

In [ ]:
# @param

import ee
import geemap

### Helper functions ### ----------------------------------------------------------------------------

def is_image_aoi(AOI) -> bool:
    """Return True if AOI is an ee.Image mask (as opposed to a vector AOI)."""
    return isinstance(AOI, ee.image.Image)


def normalize_aoi(AOI):
    """Normalize an AOI input into a (mask_image, geometry) tuple.

    The workflow expects an AOI *image mask* (so it can call `.updateMask(AOI)` and
    `.geometry()`). For vector AOIs (Geometry / Feature / FeatureCollection), this
    function generates an equivalent constant mask image.

    Supported inputs:
    - `ee.Image`: non-zero pixels are treated as inside the AOI
    - `ee.Geometry`, `ee.Feature`, `ee.FeatureCollection`

    Returns:
        (ee.Image, ee.Geometry):
            - mask image (value 1 inside AOI, masked outside)
            - AOI geometry
    """

    # Image AOI: treat non-zero pixels as inside
    if is_image_aoi(AOI):
        aoi_img = ee.Image(AOI)
        first_band = ee.String(aoi_img.bandNames().get(0))
        inside = aoi_img.select([first_band]).neq(0)
        mask_img = inside.selfMask().rename('AOI')
        return mask_img, aoi_img.geometry()

    # Vector AOI: Geometry / Feature / FeatureCollection
    if isinstance(AOI, ee.geometry.Geometry):
        geom = ee.Geometry(AOI)
    elif isinstance(AOI, ee.feature.Feature):
        geom = ee.Feature(AOI).geometry()
    elif isinstance(AOI, ee.featurecollection.FeatureCollection):
        geom = ee.FeatureCollection(AOI).geometry()
    else:
        # Best-effort cast (e.g. table asset id / ComputedObject representing a collection)
        geom = ee.FeatureCollection(AOI).geometry()

    mask_img = ee.Image.constant(1).clip(geom).selfMask().rename('AOI')
    return mask_img, geom


def filter_by_year_and_month(COL, YEAR, MONTH):
    """Filter an Earth Engine ImageCollection by a specific year and month.

    Args:
        COL (ee.ImageCollection): Collection to filter.
        YEAR (int): Year to filter by.
        MONTH (int): Month to filter by (1-12).

    Returns:
        ee.ImageCollection: Filtered collection containing images from the specified year and month.
    """
    start_date = ee.Date.fromYMD(YEAR, MONTH, 1)
    end_date = start_date.advance(1, 'month')
    return COL.filterDate(start_date, end_date)


def apply_cloud_filtering(COL, ROI_GEO, DATE_START, DATE_END, CLOUD_THRESHOLD=0.6):
    """Apply Cloud Score+ filtering to a Sentinel-2-based ImageCollection.

    This uses the `GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED` collection and masks
    out pixels with CloudScore+ `cs` below the provided threshold.

    Args:
        COL (ee.ImageCollection): Input collection (must be linkable to the csPlus collection).
        ROI_GEO (ee.Geometry): Region of interest used to spatially filter CloudScore+.
        DATE_START (str or ee.Date): Start date (inclusive).
        DATE_END (str or ee.Date): End date (exclusive).
        CLOUD_THRESHOLD (float, optional): Minimum clear-sky score to keep (0-1). Defaults to 0.6.

    Returns:
        ee.ImageCollection: Cloud-filtered image collection.
    """

    def apply_cloud_mask(img):
        return img.updateMask(img.select('cs').gte(CLOUD_THRESHOLD))

    cs_plus = (
        ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
        .filterBounds(ROI_GEO)
        .filterDate(DATE_START, DATE_END)
        .select('cs')
    )

    return COL.linkCollection(cs_plus, ['cs']).map(apply_cloud_mask)


def calc_water(img):
    """Compute NDWI and a Dynamic-World-based water mask.

    The function:
    - computes NDWI from Sentinel-2 bands B3 (green) and B8 (NIR)
    - computes a polynomial NDWI-dependent threshold
    - classifies pixels as water when Dynamic World `water` probability >= threshold

    Args:
        img (ee.Image): Image containing bands 'B3', 'B8', and the Dynamic World band 'water'.

    Returns:
        ee.Image: Input image with two additional bands:
            - 'NDWI' (float)
            - 'water_mask' (0/1, masked where `water` is masked)
    """

    ndwi = img.normalizedDifference(['B3', 'B8']).add(1).divide(2).rename('NDWI')

    water_prob = img.select('water')

    # Polynomial threshold based on NDWI
    ndwi_val = ndwi.select('NDWI')
    ndwi_sq = ndwi_val.multiply(ndwi_val)
    threshold = (
        ndwi_val.multiply(-1.82940789473684)
        .add(ndwi_sq.multiply(0.867083333333334))
        .add(1.08532894736842)
    )

    water_mask = water_prob.gte(threshold).rename('water_mask').mask(water_prob.gte(0))

    return img.addBands(water_mask).addBands(ndwi)

def load_fabdem(ROI_GEO, smooth=True):
    """Load FABDEM and optionally apply Gaussian smoothing.

    Args:
        ROI_GEO (ee.Geometry): Region of interest.
        smooth (bool, optional): If True, apply Gaussian smoothing. Defaults to True.

    Returns:
        ee.Image: FABDEM elevation image (meters), clipped to ROI_GEO and reprojected to EPSG:3857 @ 30 m.
    """

    fabdem = ee.ImageCollection("projects/sat-io/open-datasets/FABDEM")
    fabdem_elev = (
        fabdem.filterBounds(ROI_GEO)
        .mosaic()
        .clip(ROI_GEO)
        .reproject(crs='EPSG:3857', scale=30)
    )

    smoothed_dem = fabdem_elev.convolve(
        ee.Kernel.gaussian(
            radius=60,  # meters
            sigma=30,   # meters
            units='meters',
        )
    )

    return smoothed_dem if smooth else fabdem_elev


def model_shadow_mask(DEM, AZIMUTH, ZENITH, expand=True):
    """Generate a terrain shadow mask from a DEM.

    This uses `ee.Terrain.hillShadow` with a Mercator projection (EPSG:3857), as
    required by the Earth Engine documentation.

    Args:
        DEM (ee.Image): Elevation image.
        AZIMUTH (float or ee.Number): Sun azimuth angle (degrees).
        ZENITH (float or ee.Number): Sun zenith angle (degrees).
        expand (bool, optional): If True, dilate the shadow mask slightly. Defaults to True.

    Returns:
        ee.Image: Single-band Byte image named 'monthly_shadow_mask' (1 = shadow, 0 = not shadow).
    """

    sun_azimuth = ee.Number(AZIMUTH)
    sun_zenith = ee.Number(ZENITH)

    shadow_areas = (
        ee.Terrain.hillShadow(
            DEM.reproject(crs='EPSG:3857', scale=30),
            sun_azimuth,
            sun_zenith,
            100,   # neighborhood size
            True,  # hysteresis
        )
        .eq(0)
        .unmask(0)
    )

    if expand:
        shadow_areas = shadow_areas.focal_max(2, 'circle', 'pixels').unmask(0)

    return shadow_areas.rename('monthly_shadow_mask')


def load_s2_dw_water(ROI, YEAR_MIN, YEAR_MAX, FILTER_CLOUDS=True, CLOUD_THRESHOLD=0.60):
    """Load Sentinel-2 + Dynamic World water observations for a time range.

    Dynamic World provides a per-observation water probability band (`water`) and
    a discrete label (`label`). This function links each Dynamic World observation
    with the matching Sentinel-2 image to retrieve B3/B8 (for NDWI), then derives
    `NDWI` and `water_mask` bands.

    Args:
        ROI (ee.Geometry): Region of interest geometry.
        YEAR_MIN (int or str): Start year (inclusive).
        YEAR_MAX (int or str): End year (inclusive).
        FILTER_CLOUDS (bool, optional): If True, apply CloudScore+ masking. Defaults to True.
        CLOUD_THRESHOLD (float, optional): Minimum CloudScore+ `cs` to keep (0-1). Defaults to 0.60.

    Returns:
        ee.ImageCollection: Collection containing the bands:
            - 'water' (Dynamic World water probability)
            - 'label' (Dynamic World label)
            - 'NDWI'
            - 'water_mask'
    """

    def set_has_data(img):
        """Set a boolean 'has_data' property based on pixel count inside ROI."""
        count = (
            img.reduceRegion(
                reducer=ee.Reducer.count(),
                geometry=ROI,
                scale=10,
                maxPixels=1e13,
            )
            .values()
            .get(0)
        )
        has_data = ee.Algorithms.If(ee.Number(count).gt(0), True, False)
        return img.set('has_data', has_data)

    def get_s2_image(dw_img):
        """Link a Dynamic World observation to its matching Sentinel-2 image."""
        index = dw_img.get('system:index')

        sr_img = S2_SR.filter(ee.Filter.eq('system:index', index)).first()
        toa_img = S2_TOA.filter(ee.Filter.eq('system:index', index)).first()

        s2_img = ee.Image(ee.Algorithms.If(sr_img, sr_img, toa_img))

        return (
            dw_img.addBands(s2_img.select(['B3', 'B8']))
            .set({
                'MEAN_SOLAR_AZIMUTH_ANGLE': s2_img.get('MEAN_SOLAR_AZIMUTH_ANGLE'),
                'MEAN_SOLAR_ZENITH_ANGLE': s2_img.get('MEAN_SOLAR_ZENITH_ANGLE')
            })
        )

    # Date filter
    date_min = YEAR_MIN + '-01-01'
    START = ee.Date(date_min)
    date_max = YEAR_MAX + '-01-01'
    END = ee.Date(date_max).advance(1,'year')

    col_filter = ee.Filter.And(
        ee.Filter.bounds(ROI.simplify(100)),
        ee.Filter.date(START, END),
    )

    S2_SR = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filter(col_filter).select(['B3', 'B8'])
    DW = ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1').filter(col_filter).select(['water', 'label'])

    if int(YEAR_MIN) < 2019:
        S2_TOA = ee.ImageCollection('COPERNICUS/S2_HARMONIZED').filter(col_filter).select(['B3', 'B8'])

        if int(YEAR_MIN) < 2017:
            col_filtered = DW.linkCollection(
                S2_TOA,
                ['B3', 'B8'],
                ['MEAN_SOLAR_AZIMUTH_ANGLE', 'MEAN_SOLAR_ZENITH_ANGLE'],
            )
        else:
            DW = DW.map(set_has_data).filter(ee.Filter.eq('has_data', True))
            col_filtered = DW.map(get_s2_image)

    else:
        col_filtered = DW.linkCollection(
            S2_SR,
            ['B3', 'B8'],
            ['MEAN_SOLAR_AZIMUTH_ANGLE', 'MEAN_SOLAR_ZENITH_ANGLE'],
        )

    col_water = col_filtered.map(calc_water)

    if FILTER_CLOUDS:
        col_water = apply_cloud_filtering(
            col_water,
            ROI_GEO=ROI,
            DATE_START=START,
            DATE_END=END,
            CLOUD_THRESHOLD=CLOUD_THRESHOLD,
        ).select(['water', 'label', 'NDWI', 'water_mask'])
    else:
        col_water = col_water.select(['water', 'label', 'NDWI', 'water_mask'])

    return col_water


### Main functions ### ----------------------------------------------------------------------------

def calculate_water_occurrence(
    YEAR,
    AOI,
    FILTER_CLOUDS=True,
    CLOUD_THRESHOLD=0.60,
    MASK_SHADOWS=False,
    SHADOWS_WO_THRESHOLD=40,
    INCLUDE_LABEL_MODE=False,
    LABEL_NODATA_VALUE=255,
):
    """Compute monthly and annual water occurrence frequency (WOF).

    The output image contains:
    - `WOF_1` ... `WOF_12`: monthly water occurrence frequency (%)
    - `WOF_YEAR`: annual water occurrence frequency (%)
    - `OBS_WATER`: total count of water observations (sum of monthly water detections)
    - `OBS_TOTAL`: total count of valid observations
    - optionally `DW_LABEL`: annual mode of the Dynamic World label

    Args:
        YEAR (int or str): Year to process.
        AOI: Area of interest, either an `ee.Image` mask (non-zero pixels define the AOI)
            or a vector (`ee.Geometry` / `ee.Feature` / `ee.FeatureCollection`).
        FILTER_CLOUDS (bool, optional): If True, apply CloudScore+ masking. Defaults to True.
        CLOUD_THRESHOLD (float, optional): Minimum CloudScore+ `cs` to keep (0-1). Defaults to 0.60.
        MASK_SHADOWS (bool, optional): If True, mask terrain-shadowed pixels. Defaults to False.
        SHADOWS_WO_THRESHOLD (float, optional): Shadowed pixels with annual WOF >= this threshold
            are kept (i.e., removed from the shadow mask). Defaults to 40.
        INCLUDE_LABEL_MODE (bool, optional): If True, append a `DW_LABEL` band. Defaults to False.
        LABEL_NODATA_VALUE (int, optional): Value for pixels with no Dynamic World observations in
            `DW_LABEL`. Defaults to 255.

    Returns:
        ee.Image: Image clipped to the AOI.
    """

    aoi_mask, aoi_geom = normalize_aoi(AOI)

    col_water = load_s2_dw_water(
        aoi_geom,
        YEAR,
        YEAR,
        FILTER_CLOUDS=FILTER_CLOUDS,
        CLOUD_THRESHOLD=CLOUD_THRESHOLD,
    )

    label_mode_image = None
    if INCLUDE_LABEL_MODE:
        label_collection = col_water.select('label')
        label_mode_image = ee.Image(
            ee.Algorithms.If(
                label_collection.size().gt(0),
                label_collection.mode().toByte(),
                ee.Image.constant(LABEL_NODATA_VALUE).toByte(),
            )
        ).rename('DW_LABEL')
        label_mode_image = label_mode_image.updateMask(aoi_mask).unmask(LABEL_NODATA_VALUE, False)

    def process_month(MONTH):
        """Compute monthly WOF and observation counts (server-side)."""
        monthly_col = filter_by_year_and_month(col_water, ee.Number.parse(YEAR), MONTH)

        def process_non_empty_monthly_col(non_empty_col):
            monthly_obs_count = (
                non_empty_col.select('water')
                .reduce(ee.Reducer.count())
                .rename('monthly_observation_count')
                .toFloat()
            )
            monthly_water_count = (
                non_empty_col.select('water_mask')
                .reduce(ee.Reducer.sum())
                .rename('monthly_water_count')
                .toFloat()
            )

            monthly_water_mask = monthly_water_count.gt(0).rename('monthly_water_mask').unmask(0).toByte()
            observation_mask = monthly_obs_count.gt(0).rename('monthly_observation_mask').unmask(0).toByte()

            monthly_wo = (
                monthly_water_count.divide(monthly_obs_count)
                .multiply(100)
                .rename('water_occurrence_month')
                .toFloat()
            )

            output = (
                monthly_wo.addBands(monthly_obs_count)
                .addBands(monthly_water_count)
                .addBands(monthly_water_mask)
                .addBands(observation_mask)
                .set({
                    'year': YEAR,
                    'month': MONTH,
                    'empty': 0,
                    'azimuth': non_empty_col.aggregate_mean('MEAN_SOLAR_AZIMUTH_ANGLE'),
                    'zenith': non_empty_col.aggregate_mean('MEAN_SOLAR_ZENITH_ANGLE'),
                })
                .updateMask(observation_mask)
            )

            return output

        def create_empty_monthly_image():
            """Typed placeholder for empty months (keeps collection homogeneous)."""
            wo_img = ee.Image.constant(0).toFloat().rename('water_occurrence_month')
            obs_count_img = ee.Image.constant(0).toFloat().rename('monthly_observation_count')
            water_count_img = ee.Image.constant(0).toFloat().rename('monthly_water_count')
            water_mask_img = ee.Image.constant(0).toByte().rename('monthly_water_mask')
            obs_mask_img = ee.Image.constant(0).toByte().rename('monthly_observation_mask')

            return (
                ee.Image.cat([wo_img, obs_count_img, water_count_img, water_mask_img, obs_mask_img])
                .set({'year': YEAR, 'month': MONTH, 'empty': 1, 'azimuth': 0, 'zenith': 0})
            )

        return ee.Algorithms.If(
            monthly_col.size().gt(0),
            process_non_empty_monthly_col(monthly_col),
            create_empty_monthly_image()
        )

    monthly_collection = ee.ImageCollection(
        ee.List(ee.List.sequence(1, 12).map(lambda m: process_month(m)))
    )

    water_count = monthly_collection.select('monthly_water_count').sum().rename('OBS_WATER')
    observation_count = monthly_collection.select('monthly_observation_count').sum().rename('OBS_TOTAL')

    water_month_count = monthly_collection.select('monthly_water_mask').sum().rename('water_month_count')
    total_water_mask = water_month_count.gt(0).rename('total_water_mask')

    observation_month_count = (
        monthly_collection.select('monthly_observation_mask')
        .sum()
        .rename('observation_month_count')
        .updateMask(total_water_mask)
    )

    yearly_occurrence_sum = (
        monthly_collection.select('water_occurrence_month')
        .sum()
        .rename('water_occurrence_year_sum')
        .selfMask()
    )
    yearly_occurrence = (
        yearly_occurrence_sum.divide(observation_month_count)
        .rename('WOF_YEAR')
        .updateMask(total_water_mask)
    )

    if MASK_SHADOWS:
        smoothed_dem = load_fabdem(ROI_GEO=aoi_geom.buffer(1000), smooth=True)

        def process_shadows_month(month_img):
            return ee.Algorithms.If(
                ee.Algorithms.IsEqual(month_img.get('azimuth'), None),
                ee.Image.constant(0).toByte().rename('monthly_shadow_mask').set(
                    {'year': month_img.get('year'), 'month': month_img.get('month'), 'empty': 1}
                ),
                ee.Algorithms.If(
                    ee.Algorithms.IsEqual(month_img.get('empty'), 1),
                    ee.Image.constant(0).toByte().rename('monthly_shadow_mask').set(
                        {'year': month_img.get('year'), 'month': month_img.get('month'), 'empty': 1}
                    ),
                    ee.Image(
                        model_shadow_mask(
                            DEM=smoothed_dem,
                            AZIMUTH=month_img.get('azimuth'),
                            ZENITH=month_img.get('zenith'),
                            expand=True,
                        )
                    )
                    .rename('monthly_shadow_mask')
                    .unmask(0)
                    .toByte()
                    .set({'year': month_img.get('year'), 'month': month_img.get('month'), 'empty': 0}),
                ),
            )

        monthly_shadow_collection = ee.ImageCollection(monthly_collection.map(process_shadows_month))

        def apply_shadow_mask(img):
            monthly_shadow_mask = monthly_shadow_collection.filter(
                ee.Filter.eq('month', img.get('month'))
            ).first()

            safe_shadow = ee.Algorithms.If(
                ee.Algorithms.IsEqual(monthly_shadow_mask, None),
                ee.Image.constant(0).toByte().rename('monthly_shadow_mask'),
                ee.Algorithms.If(
                    ee.List(ee.Image(monthly_shadow_mask).bandNames()).contains('monthly_shadow_mask'),
                    ee.Image(monthly_shadow_mask).select('monthly_shadow_mask').unmask(0).toByte(),
                    ee.Image.constant(0).toByte().rename('monthly_shadow_mask'),
                ),
            )
            safe_shadow = ee.Image(safe_shadow)

            # Keep pixels that are in shadow only if annual WOF is high enough.
            shadow_mask_wo_removed = safe_shadow.updateMask(
                yearly_occurrence.select('WOF_YEAR').lt(SHADOWS_WO_THRESHOLD)
            ).unmask(0)

            return img.updateMask(shadow_mask_wo_removed.eq(0))

        monthly_collection = monthly_collection.map(apply_shadow_mask)

        water_count = (
            monthly_collection.select('monthly_water_count')
            .sum()
            .rename('OBS_WATER')
            .updateMask(total_water_mask)
        )
        observation_count = (
            monthly_collection.select('monthly_observation_count')
            .sum()
            .rename('OBS_TOTAL')
            .updateMask(total_water_mask)
        )

        observation_month_count = (
            monthly_collection.select('monthly_observation_mask')
            .sum()
            .rename('observation_month_count')
            .updateMask(total_water_mask)
        )
        yearly_occurrence_sum = (
            monthly_collection.select('water_occurrence_month')
            .sum()
            .rename('water_occurrence_year_sum')
            .selfMask()
        )
        yearly_occurrence = (
            yearly_occurrence_sum.divide(observation_month_count)
            .rename('WOF_YEAR')
            .updateMask(total_water_mask)
        )

    def get_monthly_band(m):
        img = monthly_collection.filter(ee.Filter.eq('month', m)).first()

        return ee.Algorithms.If(
            ee.Number(ee.Image(img).get('empty')).eq(1),
            ee.Image.constant(0).toFloat().rename(['WOF_' + str(m)]),
            ee.Image(img).select('water_occurrence_month').rename(['WOF_' + str(m)]),
        )

    monthly_bands = ee.Image.cat([get_monthly_band(m) for m in range(1, 13)])

    result = (
        monthly_bands.addBands(yearly_occurrence)
        .addBands(water_count)
        .addBands(observation_count)
        .updateMask(aoi_mask)
    )

    if INCLUDE_LABEL_MODE and label_mode_image is not None:
        result = ee.Image.cat([result, label_mode_image])

    return result


### Export function ### ----------------------------------------------------------------------------

def get_WO_tiles_and_export(
    ROI,
    YEAR,
    FOLDER_NAME='WO_mapping',
    BASE_FILENAME='WOF',
    EXPORT_CRS='EPSG:4326',
    FILE_SCALE=10,
    FILTER_CLOUDS=True,
    CLOUD_THRESHOLD=0.60,
    MASK_SHADOWS=False,
    SHADOWS_WO_THRESHOLD=0,
    INCLUDE_LABEL_MODE=True,
):
    """Compute WOF for an AOI and export it to Google Drive.

    The exported image is converted to Byte and uses 255 as NoData.

    Args:
        ROI: AOI (same accepted types as `calculate_water_occurrence`).
        YEAR (int or str): Year to process.
        FOLDER_NAME (str, optional): Google Drive folder name. Defaults to 'WO_mapping'.
        BASE_FILENAME (str, optional): Base output filename. Defaults to 'WOF'.
        EXPORT_CRS (str, optional): Export CRS (e.g. 'EPSG:4326'). Defaults to 'EPSG:4326'.
        FILE_SCALE (int, optional): Pixel size for export in meters. Defaults to 10.
        FILTER_CLOUDS (bool, optional): Apply CloudScore+ masking. Defaults to True.
        CLOUD_THRESHOLD (float, optional): Minimum CloudScore+ `cs` to keep. Defaults to 0.60.
        MASK_SHADOWS (bool, optional): Mask terrain shadows. Defaults to False.
        SHADOWS_WO_THRESHOLD (float, optional): Shadow WOF threshold. Defaults to 0.
        INCLUDE_LABEL_MODE (bool, optional): Include `DW_LABEL` band. Defaults to True.

    Returns:
        None. Starts a Drive export task.
    """

    aoi_mask, aoi_geom = normalize_aoi(ROI)

    wof_img = calculate_water_occurrence(
        YEAR,
        AOI=aoi_mask,
        FILTER_CLOUDS=FILTER_CLOUDS,
        CLOUD_THRESHOLD=CLOUD_THRESHOLD,
        MASK_SHADOWS=MASK_SHADOWS,
        SHADOWS_WO_THRESHOLD=SHADOWS_WO_THRESHOLD,
        INCLUDE_LABEL_MODE=INCLUDE_LABEL_MODE,
        LABEL_NODATA_VALUE=255,
    )

    image_export = wof_img.toByte().unmask(255, False)
    task_name = f"{BASE_FILENAME}_{YEAR}"

    task = ee.batch.Export.image.toDrive(
        image=image_export,
        description='Export_' + task_name,
        folder=str(FOLDER_NAME).strip(),
        fileNamePrefix=task_name,
        region=aoi_geom,
        crs=EXPORT_CRS,
        scale=FILE_SCALE,
        maxPixels=1e13,
        skipEmptyTiles=True,
        formatOptions={'cloudOptimized': True, 'noData': 255},
    )

    task.start()
    print(f"Task started: {task_name} (check the Earth Engine Tasks tab for progress).")


## Step 2: Link to a Google Earth Engine project
A Google Earth Engine account and a corresponding project are required to run this workflow.

In [ ]:
# set project name of your GEE project
GEE_PROJECT = "CUSTOM_PROJECT"

# authenticate and initialize GEE by setting the project name
ee.Authenticate()
ee.Initialize(project = GEE_PROJECT)

## Step 3: Define study area and year
Provide the study area as a GEE asset (either a Feature/FeatureCollection or an Image mask). Then define the year, the output name, and the export coordinate reference system (CRS).

In [ ]:
# load Area of Interest (AOI) - can be an Image mask (1 for data, 0 for no data) or a vector geometry/Feature/FeatureCollection
AOI = ee.FeatureCollection("projects/" + GEE_PROJECT + "/assets/CUSTOM_ASSET")
YEAR = '2025'
BASE_FILENAME='WOF_YOURASSET'
CRS='EPSG:4326'

## Step 4: Visualize (*only for relatively small regions*)

In [ ]:
# Create a geemap Map
m = geemap.Map()
m.centerObject(AOI, 12)

# Compute water occurrence (can be slow for large AOIs)
wof = calculate_water_occurrence(
    YEAR=YEAR,
    AOI=AOI,
    FILTER_CLOUDS=True,
    MASK_SHADOWS=True,
    SHADOWS_WO_THRESHOLD=40,
)

# Visualization parameters
water_prob_vis = {
    'min': 1,
    'max': 100,
    'palette': ['#bddf26', '#7ad151', '#44bf70', '#22a884', '#21918c', '#2a788e', '#355f8d', '#414487', '#482475', '#440154'],
}

# Add layers
m.add_layer(AOI, {}, 'AOI')
m.add_layer(wof.select('WOF_YEAR'), water_prob_vis, 'Annual WOF (WOF_YEAR)')
# m.add_layer(wof.select('WOF_5'), water_prob_vis, 'May WOF (WOF_5)') # Example for adding a monthly WOF layer


# show map
m

## Step 5: Export to Google Drive

In [ ]:
# run the export function
get_WO_tiles_and_export(
    AOI,
    YEAR=YEAR,
    BASE_FILENAME=BASE_FILENAME,
    FOLDER_NAME=f"{BASE_FILENAME}_{YEAR}",
    FILTER_CLOUDS = True,
    MASK_SHADOWS = True,
    INCLUDE_LABEL_MODE=True,
    EXPORT_CRS=CRS
    )